In [1]:
pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
#step-1.Import required libraries:

import numpy as np
import pandas as pd
import os
import pickle

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn import metrics
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report


In [3]:
#step-2.import required dataset:

df=pd.read_csv(r"C:\Users\ELCOT\Downloads\obesity_custom_100rows.csv")
df.head()

,Gender,Age,Height_cm,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,Female,60,177.3,95.4,yes,yes,2.2,5,Always,yes,2.0,yes,2.7,0.4,Frequently,Walking,Obesity_Type_III
1,Male,59,165.3,74.2,yes,no,2.3,2,no,yes,1.4,no,0.9,0.2,Sometimes,Public_Transportation,Obesity_Type_I
2,Male,48,164.3,83.5,no,no,2.4,5,Sometimes,yes,1.3,no,2.1,1.0,Frequently,Motorbike,Obesity_Type_III
3,Male,55,156.4,125.2,yes,yes,1.1,3,Sometimes,yes,2.0,yes,2.8,0.9,Frequently,Bike,Obesity_Type_I
4,Male,30,159.0,80.6,yes,yes,1.4,2,no,no,2.4,yes,2.7,0.9,Sometimes,Public_Transportation,Overweight_Level_II


In [4]:
#drop dupicates and unwanted column:

df=df.drop_duplicates()

In [5]:
df['Age']=df['Age'].astype(int)

In [6]:
df.describe()

,Age,Height_cm,Weight,FCVC,NCP,CH2O,FAF,TUE
count,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
mean,35.900000,169.097000,87.372000,1.934000,3.030000,1.985000,1.424000,1.063000
std,12.845233,12.975762,25.916977,0.544619,1.374038,0.592269,0.860716,0.569716
min,18.000000,146.100000,45.200000,1.000000,1.000000,1.000000,0.100000,0.000000
25%,25.000000,158.350000,65.350000,1.500000,2.000000,1.400000,0.775000,0.600000
50%,33.500000,168.300000,86.700000,2.000000,3.000000,1.950000,1.300000,1.100000
75%,46.000000,181.125000,110.100000,2.400000,4.000000,2.500000,2.125000,1.600000
max,60.000000,189.900000,129.700000,3.000000,5.000000,3.000000,3.000000,1.900000


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Gender                          100 non-null    str    
 1   Age                             100 non-null    int64  
 2   Height_cm                       100 non-null    float64
 3   Weight                          100 non-null    float64
 4   family_history_with_overweight  100 non-null    str    
 5   FAVC                            100 non-null    str    
 6   FCVC                            100 non-null    float64
 7   NCP                             100 non-null    int64  
 8   CAEC                            100 non-null    str    
 9   SMOKE                           100 non-null    str    
 10  CH2O                            100 non-null    float64
 11  SCC                             100 non-null    str    
 12  FAF                             100 non-null    

In [8]:
df.isna().sum()

Gender                            0
Age                               0
Height_cm                         0
Weight                            0
family_history_with_overweight    0
FAVC                              0
FCVC                              0
NCP                               0
CAEC                              0
SMOKE                             0
CH2O                              0
SCC                               0
FAF                               0
TUE                               0
CALC                              0
MTRANS                            0
NObeyesdad                        0
dtype: int64

In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
#define numeric columns and category columns:

numeric_col=['Age','Height_cm','Weight','FCVC','NCP','CH2O','FAF','TUE']
categorical_col=['Gender','family_history_with_overweight','FAVC','CAEC','SMOKE','SCC','CALC','MTRANS']

In [11]:
#CREATE DIRESCTORY TO STORE ENCODERS AND PICKLES:

os.makedirs("model_assets",exist_ok=True)

In [12]:
#scale mumeric columns:

scale=StandardScaler()
df[numeric_col]=scale.fit_transform(df[numeric_col])

pickle.dump(scale,open('model_assets/scaler.pkl','wb'))

In [13]:
#encode categorical columns:

label_encoder = {}

for col in categorical_col:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoder[col] = le

    pickle.dump(le,open(f'model_assets/{col}_lebelEncoder.pkl','wb'))

In [14]:
#encode target column:

target_encoder=LabelEncoder()

df['NObeyesdad']=target_encoder.fit_transform(df['NObeyesdad'])

pickle.dump(target_encoder,open(f'model_assets/{col}_lebelEncoder.pkl','wb'))

In [15]:
#split features and target:

x=df.drop(columns='NObeyesdad')
y=df['NObeyesdad']
x.shape,y.shape

((100, 16), (100,))

In [16]:
#train test split:

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=21)
display(x_train.shape,y_train.shape,x_test.shape,y_test.shape)

(80, 16)

(80,)

(20, 16)

(20,)

In [17]:
#grid search for better model:

param_grid={
    'n_neighbors': list(range(1,50)),
    'weights': ['uniform','distance'],   # ← comma சேர்க்கவும்
    'metric': ['euclidean','manhattan']
}

knn=KNeighborsClassifier()
grid_search=GridSearchCV(knn,param_grid,cv=5,scoring='accuracy',verbose=1,n_jobs=-1)
grid_search.fit(x_train,y_train)

Fitting 5 folds for each of 196 candidates, totalling 980 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",KNeighborsClassifier()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'metric': ['euclidean', 'manhattan'], 'n_neighbors': [1, 2, ...], 'weights': ['uniform', 'distance']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed

In [18]:
# best model :

best_knn = grid_search.best_estimator_

In [19]:
#evaluate:

y_pred=best_knn.predict(x_test)
print(y_pred)

[1 6 4 1 6 4 6 1 1 1 5 1 1 4 4 2 1 6 1 1]


In [20]:
con_mat=metrics.confusion_matrix(y_test,y_pred)
print("Confusion Matrix:\n",con_mat)

Confusion Matrix:
 [[0 1 0 0 1 0 1]
 [0 1 0 0 1 0 0]
 [0 2 1 0 0 1 0]
 [0 2 0 0 0 0 1]
 [0 1 0 0 0 0 1]
 [0 3 0 0 2 0 1]
 [0 0 0 0 0 0 0]]


In [21]:
accu_score=metrics.accuracy_score(y_test,y_pred)
print("Accuracy Score:\n",accu_score)
print("Accuracy Score:\n",accu_score*100,"%")

Accuracy Score:
 0.1
Accuracy Score:
 10.0 %


In [22]:
report=classification_report(y_test,y_pred)
print("Classification Report:\n",report)

Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.10      0.50      0.17         2
           2       1.00      0.25      0.40         4
           3       0.00      0.00      0.00         3
           4       0.00      0.00      0.00         2
           5       0.00      0.00      0.00         6
           6       0.00      0.00      0.00         0

    accuracy                           0.10        20
   macro avg       0.16      0.11      0.08        20
weighted avg       0.21      0.10      0.10        20



C:\Users\ELCOT\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\ELCOT\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\ELCOT\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classificati

In [23]:
pickle.dump(best_knn,open("model_assets/obesity_KNN.pkl",'wb'))

In [28]:
Age = float(input("Enter Age: "))
Height_cm = float(input("Enter Height (cm): "))
Weight = float(input("Enter Weight (kg): "))
NCP = float(input("Enter meals per day: "))
FCVC = float(input("Enter vegetable frequency (1-3): "))
CH2O = float(input("Enter water intake (liters): "))
FAF = float(input("Enter physical activity (0-3): "))
TUE = float(input("Enter tech use per day (hours): "))

# Categorical inputs
Gender = input("Enter Gender (Male/Female): ").strip().capitalize()
Gender = 1 if Gender == 'Male' else 0

family_history = input("Family history overweight? (yes/no): ").strip().lower()
family_history_with_overweight = 1 if family_history == 'yes' else 0

FAVC = input("Enter FAVC (yes/no): ").strip().lower()
FAVC = 1 if FAVC == 'yes' else 0

CAEC = input("Enter CAEC (no/Sometimes/Frequently/Always): ").strip()
CAEC_map = {'no': 0, 'Sometimes': 1, 'Frequently': 2, 'Always': 3}
CAEC = CAEC_map[CAEC]

SMOKE = input("Enter SMOKE (yes/no): ").strip().lower()
SMOKE = 1 if SMOKE == 'yes' else 0

SCC = input("Enter SCC (yes/no): ").strip().lower()
SCC = 1 if SCC == 'yes' else 0

CALC = input("Enter CALC (no/Sometimes/Frequently/Always): ").strip()
CALC_map = {'no': 0, 'Sometimes': 1, 'Frequently': 2, 'Always': 3}
CALC = CALC_map[CALC]

MTRANS = input("Enter Transport (Public_Transportation/Walking/Automobile/Motorbike/Bike): ").strip()
MTRANS_map = {'Public_Transportation': 0, 'Walking': 1, 'Automobile': 2, 'Motorbike': 3, 'Bike': 4}
MTRANS = MTRANS_map[MTRANS]

# Prediction
import pickle
p = pickle.load(open('model_assets/obesity_KNN.pkl', 'rb'))
result = p.predict([[Gender, Age, Height_cm, Weight,
                     family_history_with_overweight,
                     FAVC, FCVC, NCP, CAEC, SMOKE,
                     CH2O, SCC, FAF, TUE, CALC, MTRANS]])

print("Predicted Obesity Level:", result[0])

Predicted Obesity Level: 1


C:\Users\ELCOT\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(
